# Gemma 4 (E2B) QLoRA 微调 — 日报数据集

**打开 Colab → 按顺序运行每个 cell。**

| 项目 | 详情 |
|------|------|
| 模型 | `google/gemma-4-2b-it` (E2B, 2B 参数) |
| 方法 | QLoRA (4-bit) |
| 数据 | 984 条 ChatML JSONL (2025-2026 日报) |
| 显存 | ~2 GB (T4 15GB 完全够) |
| 训练 | ~120 steps / 15-25 分钟 |

## 第 1 步：安装 Unsloth

In [ ]:
%%capture
!pip install unsloth

## 第 2 步：检查数据集

确保 `gemma_finetune_dataset.jsonl` 在当前目录下。

In [ ]:
import os

data_file = "gemma_finetune_dataset.jsonl"
if not os.path.exists(data_file):
    print(f"❌ 找不到数据集文件: {data_file}")
    print(f"请把文件放到当前目录: {os.getcwd()}")
else:
    file_size = os.path.getsize(data_file) / 1024
    print(f"✅ 已找到数据集文件: {data_file} ({file_size:.1f} KB)")

## 第 3 步：加载模型 (QLoRA 4-bit)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 512
dtype = None
load_in_4bit = True

MODEL_NAME = "google/gemma-4-2b-it"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print(f"模型: {MODEL_NAME}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"显存: {round(torch.cuda.get_device_properties(0).total_memory/1024**3, 1)} GB")

## 第 4 步：Chat Template + 加载数据集

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)

def formatting_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        )
        for convo in convos
    ]
    return {"text": texts}

from datasets import load_dataset
dataset = load_dataset(
    "json",
    data_files = "gemma_finetune_dataset.jsonl",
    split = "train"
)
dataset = dataset.map(formatting_func, batched=True)

print(f"数据集: {len(dataset)} 条")
print("第一条:", dataset[0]["text"][:200] + "...")

## 第 5 步：配置 LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

model.print_trainable_parameters()

## 第 6 步：配置 Trainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,
        warmup_steps = 6,
        max_steps = 120,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

print(f"有效 batch: 2×8=16 | 步数: 120 | lr: 2e-4")

## 第 7 步：只训练 assistant 回复

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part    = "<start_of_turn>model\n",
)
print("已启用")

## 第 8 步：开始训练

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_mem = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
max_mem = round(gpu_stats.total_memory / 1024**3, 3)
print(f"{gpu_stats.name} | {max_mem} GB | 已用 {start_mem} GB")

trainer_stats = trainer.train()

In [ ]:
used = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
print(f"final loss: {trainer_stats.metrics['train_loss']:.4f}")
print(f"显存峰值: {used} GB / {max_mem} GB ({round(used/max_mem*100,1)}%)")

## 第 9 步：保存并下载

In [ ]:
# LoRA 适配器 (~50 MB)
model.save_pretrained("gemma4_diary_lora")
tokenizer.save_pretrained("gemma4_diary_lora")

# 合并 16-bit 完整模型
model.save_pretrained_merged("gemma4_diary_merged", tokenizer, save_method="merged_16bit")

# GGUF (Ollama 本地用)
model.save_pretrained_gguf("gemma4_diary_gguf", tokenizer, quantization_method="q4_k_m")

print("全部保存完成")

In [ ]:
import os

lora_dir = "gemma4_diary_lora"
if os.path.exists(lora_dir):
    print(f"✅ LoRA 适配器已保存到: {os.path.abspath(lora_dir)}/")
else:
    print("⚠️ LoRA 适配器目录不存在，请先运行上一个 cell 保存模型。")

## 第 10 步：测试推理

In [ ]:
FastLanguageModel.for_inference(model)

for q in ["我在2025年8月9日做了什么？", "我的主要业务是什么？", "我当时是怎么看待过程和结果的？"]:
    messages = [{"role": "user", "content": q}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    outputs = model.generate(input_ids=inputs, max_new_tokens=512, temperature=0.7, do_sample=True, use_cache=True)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\n{'='*60}\nQ: {q}\n{'='*60}\nA: {response}")

---
## 附录

| 问题 | 解决 |
|------|------|
| loss 不降 (>1.0) | `max_steps = 180`, `lr = 4e-4` |
| 过拟合 (<0.2) | `max_steps = 60`, `lr = 1e-4`, `lora_dropout = 0.05` |
| CUDA OOM | `batch = 1`, `grad_accum = 16`, `seq_len = 256` |